In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/indotoxic2024_annotated_data-3.csv")

In [2]:
# hapus spam
df = df[df["is_noise_or_spam_text"] == 0].copy()

# hapus teks yang kosong
empty_mask = (
    df["text"].isna() |
    df["text"].astype(str).str.strip().eq("")
)
df = df[~empty_mask].copy()

### Lowercase

In [3]:
def lowercase_text(text):
    if pd.isna(text):
        return ""
    
    return str(text).lower()

df["clean_text"] = df["text"].apply(lowercase_text)

### URL & Mention Handling

In [4]:
import re

def replace_url_mention(text):
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"@\w+", " MENTION ", text)
    
    return text

df["clean_text"] = df["clean_text"].apply(replace_url_mention)

### Ekstrak Emoji

In [5]:
import emoji

def extract_emojis(text):
    if pd.isna(text):
        return ""
    
    return " ".join(
        char["emoji"]
        for char in emoji.emoji_list(str(text))
    )

df["emoji"] = df["text"].apply(extract_emojis)

In [6]:
df["emoji"].nunique()

2434

### Duplicate Handling

In [8]:
# Mengambil teks duplikat
duplicate_mask = df.duplicated(
    subset=["text"],
    keep=False
)

duplicates = df[duplicate_mask].copy()

# Mengambil teks yang memiliki label konflik
label_conflict = (
    df.groupby("text")["toxicity"]
      .nunique()
      .reset_index(name="n_label")
)

conflict_texts = label_conflict[
    label_conflict["n_label"] > 1
]["text"]

# simpan konflik ke file CSV utk dokumentasi
df_conflict = df[
    df["text"].isin(conflict_texts)
].copy()

df_conflict.to_csv(
    "../data/interim/label_conflict.csv",
    index=False
)

# hapus teks yang memiliki label konflik
df = df[
    ~df["text"].isin(conflict_texts)
].copy()

df = df.drop_duplicates(
    subset=["text"],
    keep="first"
).copy()

In [9]:
print("Shape akhir setelah cleaning:")
print(df.shape)

print("\nDuplicate text:")
print(df["text"].duplicated().sum())

print("\nMissing text:")
print(df["text"].isna().sum())

Shape akhir setelah cleaning:
(23009, 19)

Duplicate text:
0

Missing text:
0


### Topic Handling

In [10]:
df["topic_list"] = (
    df["topic"]
    .fillna("")
    .apply(
        lambda x: [
            topic.strip()
            for topic in str(x).split(",")
            if topic.strip()
        ]
    )
)

In [11]:
df[["topic", "topic_list"]].head()

,topic,topic_list
0,Disabilitas,[Disabilitas]
1,Jewish,[Jewish]
2,"Terpolarisasi, Tionghoa","[Terpolarisasi, Tionghoa]"
3,"Terpolarisasi, Jewish","[Terpolarisasi, Jewish]"
6,"Terpolarisasi, Tionghoa","[Terpolarisasi, Tionghoa]"
